# H-N100: Tversky+CE + component-centered sampling

H-N00에서 독립적으로 시작하여 H-N80 loss와 H-N90 sampling만 결합합니다.


In [1]:
from pathlib import Path
import json, subprocess, time
import pandas as pd
from IPython.display import clear_output, display

ROOT=Path("/home/stu03/workspace/model")
DATA_ROOT=Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
PYTHON=ROOT/".venv/bin/python"
CONFIG=ROOT/"configs/experiments/hemorrhage/H-N100.yaml"
RUN_DIR=ROOT/"runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTverskyComponentV1__3d_fullres/fold_0"
PROGRESS=RUN_DIR/"progress.json"
LAUNCH_DIR=ROOT/"runs/batches/H-N100"
LAUNCH_DIR.mkdir(parents=True,exist_ok=True)


## 1. 설정·테스트·plans 확인


In [2]:
subprocess.run([str(PYTHON),"-m","pytest","-q","tests/test_h_n80_h_n90.py"],cwd=ROOT,check=True)
subprocess.run([str(PYTHON),"-c","from brainon_ai.nnunet.config import load_config; load_config('configs/experiments/hemorrhage/H-N100.yaml'); print('H-N100 config OK')"],cwd=ROOT,check=True)
subprocess.run([str(PYTHON),"scripts/preprocessing/prepare_h_n100_plans.py","--preprocessed-root",str(DATA_ROOT/"nnUNet_preprocessed")],cwd=ROOT,check=True)


...                                                                      [100%]
3 passed in 0.92s
H-N100 config OK
[ok] /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset001_ICHBinary/ICHPlansTverskyComponentV1.json


CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', 'scripts/preprocessing/prepare_h_n100_plans.py', '--preprocessed-root', '/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed'], returncode=0)

## 2. 백그라운드 학습 시작

처음은 RESUME=False, 중단된 checkpoint에서 재개할 때만 True로 바꿉니다.


In [3]:
RESUME=False
GPU="0"

resolved=LAUNCH_DIR/"H-N100.resolved.yaml"
import yaml
cfg=yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
cfg["experiment"]["continue_training"]=RESUME
resolved.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False),encoding="utf-8")

command=[str(PYTHON),"-m","brainon_ai.nnunet.runner",
         "--config",str(resolved),"--data-root",str(DATA_ROOT),
         "--results-root",str(ROOT/"runs/nnunet"),"--no-validation"]
env=dict(__import__("os").environ)
env.update({"CUDA_VISIBLE_DEVICES":GPU,"nnUNet_n_proc_DA":"0","nnUNet_def_n_proc":"1"})
log=(LAUNCH_DIR/"H-N100.log").open("a",encoding="utf-8")
process=subprocess.Popen(command,cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
(LAUNCH_DIR/"launcher.pid").write_text(str(process.pid))
print("PID:",process.pid)
print("log:",LAUNCH_DIR/"H-N100.log")


PID: 127480
log: /home/stu03/workspace/model/runs/batches/H-N100/H-N100.log


## 3. 실시간 진행률


In [4]:
def read_json(path):
    try:return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError,json.JSONDecodeError):return {}
def fmt(seconds):
    if seconds is None:return "-"
    seconds=int(seconds);h,r=divmod(seconds,3600);m,s=divmod(r,60)
    return f"{h:02d}:{m:02d}:{s:02d}"
while True:
    clear_output(wait=True)
    p=read_json(PROGRESS)
    row={"실험":"H-N100","상태":p.get("status","waiting"),
         "Epoch":f"{p.get('current_epoch',0)}/{p.get('total_epochs','-')}",
         "진행률(%)":p.get("progress_percent",0),"ETA":fmt(p.get("eta_seconds")),
         "Dice":p.get("pseudo_dice"),"Recall":p.get("pseudo_recall"),
         "Best Composite":p.get("best_composite")}
    display(pd.DataFrame([row]))
    if row["상태"] in {"completed","early_stopped","failed"}:break
    time.sleep(10)


,실험,상태,Epoch,진행률(%),ETA,Dice,Recall,Best Composite
0,H-N100,early_stopped,229/300,76.33,00:35:58,None,None,0.854605


## 4. 학습 완료 후 full-volume validation


In [5]:
status=read_json(PROGRESS).get("status")
if status not in {"completed","early_stopped"}:
    raise RuntimeError(f"학습이 아직 완료되지 않았습니다: {status}")
subprocess.run([str(PYTHON),"scripts/validation/complete_h_n100_validation.py"],cwd=ROOT,check=True)


Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##############################################################

CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', 'scripts/validation/complete_h_n100_validation.py'], returncode=0)

## 5. N00·N80·N90·N100 동일 기준 지표


In [6]:
import nibabel as nib
import numpy as np
import yaml

EXPERIMENTS=["H-N00","H-N80","H-N90","H-N100"]
OUT=ROOT/"runs/comparisons/H-N00_H-N100_final"
OUT.mkdir(parents=True,exist_ok=True)
rows=[]
for name in EXPERIMENTS:
    cfg=yaml.safe_load((ROOT/f"configs/experiments/hemorrhage/{name}.yaml").read_text())
    e,plans=cfg["experiment"],cfg["plans"]
    dataset=f"Dataset{int(e['dataset_id']):03d}_{e.get('dataset_name','ICHBinary')}"
    fold=ROOT/"runs/nnunet"/dataset/f"nnUNetTrainerICHBaseline__{plans['output_identifier']}__{e['configuration']}"/f"fold_{e['fold']}"
    summary=json.loads((fold/"validation/summary.json").read_text())
    gt_dir=DATA_ROOT/"nnUNet_preprocessed"/dataset/"gt_segmentations"
    for item in summary["metric_per_case"]:
        filename=Path(str(item["prediction_file"]).replace(chr(92),"/")).name
        case=filename[:-7] if filename.endswith(".nii.gz") else Path(filename).stem
        m=item["metrics"]["1"];tp,fp,fn=(float(m[k]) for k in ("TP","FP","FN"));nref=float(m["n_ref"])
        spacing=nib.load(gt_dir/f"{case}.nii.gz").header.get_zooms()[:3]
        volume=nref*float(np.prod(spacing))/1000;positive=nref>0
        rows.append({"experiment":name,"case_id":case,"positive":positive,"volume_ml":volume,
                     "dice":float(m["Dice"]),"recall":tp/(tp+fn) if tp+fn else np.nan,
                     "precision":tp/(tp+fp) if tp+fp else np.nan,"tp":tp,"fp":fp,"fn":fn,
                     "detected":tp>0,"true_negative":(not positive and fp==0)})
case_metrics_EOF=pd.DataFrame(rows)
case_metrics_EOF.to_csv(OUT/"per_case_metrics.csv",index=False)


In [7]:
summary_rows=[]
for name in EXPERIMENTS:
    e=case_metrics_EOF[case_metrics_EOF.experiment==name]
    pos=e[e.positive];small=pos[pos.volume_ml<5];neg=e[~e.positive]
    summary_rows.append({
        "experiment":name,"positive_cases":len(pos),
        "macro_dice":pos.dice.mean(),"macro_recall":pos.recall.mean(),
        "macro_precision":pos.precision.mean(),"detection_rate":pos.detected.mean(),
        "zero_dice_cases":int(pos.dice.fillna(0).eq(0).sum()),
        "under5_dice":small.dice.mean(),"under5_recall":small.recall.mean(),
        "under5_detection":small.detected.mean(),
        "negative_specificity":neg.true_negative.mean(),"mean_fp_voxels_negative":neg.fp.mean()})
leaderboard=pd.DataFrame(summary_rows).sort_values("macro_dice",ascending=False)
leaderboard.to_csv(OUT/"leaderboard.csv",index=False)
display(leaderboard.style.format({c:"{:.4f}" for c in leaderboard.columns if c not in {"experiment","positive_cases","zero_dice_cases"}}))
print("saved:",OUT)


,experiment,positive_cases,macro_dice,macro_recall,macro_precision,detection_rate,zero_dice_cases,under5_dice,under5_recall,under5_detection,negative_specificity,mean_fp_voxels_negative
1,H-N80,35,0.6099,0.6247,0.6447,0.9429,2,0.3675,0.4252,0.8182,0.4000,327.6000
2,H-N90,35,0.6085,0.6212,0.6404,0.9143,3,0.3733,0.4509,0.7273,0.6000,25.2000
0,H-N00,35,0.5946,0.6212,0.6068,0.9143,3,0.3447,0.4612,0.7273,0.0000,2037.6000
3,H-N100,35,0.5703,0.6451,0.5495,0.9143,3,0.3147,0.4842,0.7273,0.2000,4013.2000


saved: /home/stu03/workspace/model/runs/comparisons/H-N00_H-N100_final
